# Overflow-safe sigmoid kernel

The logistic sigmoid is

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

In `float32`, $e^{v}$ overflows once $v > 88.7$. The textbook formula evaluates $e^{-x}$, which overflows for every $x < -88.7$. On a GPU that intermediate quietly becomes `inf` (the final result happens to round to 0, but the rearranged form $e^{x}/(1+e^{x})$ would give `inf / inf = NaN`); here, on the simulator, it raises `OverflowError`. The fix is to branch on the sign so the exponent is always $\le 0$:

$$\sigma(x) = \begin{cases} \dfrac{1}{1 + e^{-x}} & x \ge 0 \\[2mm] \dfrac{e^{x}}{1 + e^{x}} & x < 0 \end{cases}$$

Write `sigmoid(x)` for a 1-D `float32` array of length `n` (possibly 0) and return a `float32` array of shape `(n,)`.

## Requirements
- One thread per element, 128 threads per block, $\lceil n/128 \rceil$ blocks; return an empty array for `n == 0` without launching.
- Use `math.exp` inside the kernel and never call it with an argument greater than 0.
- Inputs may be as large as $\pm 1000$. Results are compared with a relative tolerance of $10^{-5}$.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import math
import numpy as np
from numba import cuda

TPB = 128

def sigmoid(x):
    """x: float32 (n,). Returns 1 / (1 + exp(-x)) as float32 (n,)."""
    n = x.shape[0]
    if n == 0:
        return np.zeros(0, dtype=np.float32)
    d_x = cuda.to_device(x)
    d_out = cuda.device_array(n, dtype=np.float32)
    sigmoid_kernel[(n + TPB - 1) // TPB, TPB](d_x, d_out)
    return d_out.copy_to_host()

@cuda.jit
def sigmoid_kernel(x, out):
    i = cuda.grid(1)
    if i < x.shape[0]:
        out[i] = 1.0 / (1.0 + math.exp(-x[i]))   # TODO: overflows for very negative x


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [0.0, 1.0, -1.0, 4.0], "dtype": "float32"}], "e": {"__nd__": [0.5, 0.7310585975646973, 0.2689414322376251, 0.9820137619972229], "dtype": "float32"}}, {"a": [{"__nd__": [-1000.0, -90.0, 90.0, 1000.0], "dtype": "float32"}], "e": {"__nd__": [0.0, 8.194008692231508e-40, 1.0, 1.0], "dtype": "float32"}}, {"a": [{"__nd__": [], "dtype": "float32"}], "e": {"__nd__": [], "dtype": "float32"}}, {"a": [{"__nd__": [-88.0, -50.0, -20.0, 20.0, 50.0], "dtype": "float32"}], "e": {"__nd__": [6.054601485195952e-39, 1.9287498933537385e-22, 2.06115369216775e-09, 1.0, 1.0], "dtype": "float32"}}, {"a": [{"__nd__": [-3.258955717086792, -0.8735864758491516, 8.318619728088379, 3.295738697052002, -8.206986427307129, -0.026016321033239365, -3.117318630218506, 0.7431576251983643, -8.040939331054688, 1.2088594436645508, 1.176904559135437, 7.878129959106445, 1.583225131034851, 2.5527334213256836, -7.465583324432373, 11.26364517211914, -9.578227996826172, 5.50900936126709, -1.6495203971862793, -4.403232574462891, -3.2814249992370605, -3.3600733280181885, 1.9009454250335693, -0.550293505191803, 7.412855625152588, -9.148019790649414, -0.015404823236167431, -4.4602789878845215, 3.879359245300293, -10.590300559997559, -1.7184717655181885, 1.050864577293396, -7.421342849731445, 4.926077842712402, 0.8935671448707581, 5.034283638000488, 4.795924186706543, -4.899272918701172, -3.988978862762451, -1.0166624784469604, 3.7397985458374023, 4.254836559295654, -3.548424243927002, -3.0359859466552734, -3.98911452293396, -2.9211912155151367, -1.1896169185638428, -0.6590752601623535, 10.290572166442871, -2.5320181846618652, -1.4436218738555908, 2.292888641357422, -4.7653703689575195, -1.8431357145309448, 0.06659240275621414, 3.870729684829712, -6.580074310302734, 6.85734748840332, -1.7622867822647095, 0.8470820784568787, 4.2354416847229, 3.303964614868164, 5.296159267425537, 0.865989089012146, -0.0980667695403099, 1.582392692565918, -4.978577136993408, 6.070936679840088, -3.8769311904907227, -6.301219463348389, 10.28204345703125, -0.682224690914154, -5.895624160766602, 9.261348724365234, -1.6485631465911865, 5.3103790283203125, -4.147443771362305, -1.2439000606536865, -8.442646980285645, -9.544696807861328, -4.88217306137085, -0.3901951014995575, 3.9899284839630127, 12.131102561950684, -5.15931510925293, 5.045201301574707, -3.02170467376709, -0.7626465559005737, -7.40595817565918, -7.394928455352783, -3.892302989959717, -5.965553283691406, -0.017771953716874123, -3.5125722885131836, 6.253485202789307, 0.7650985717773438, 0.8261909484863281, 5.68717622756958, 2.5905613899230957, -0.1866649091243744, -0.45361679792404175, -0.6047431826591492, 5.383609771728516, 3.9154281616210938, 1.82958984375, 3.9941465854644775, -2.7226665019989014, 2.5551140308380127, -7.061668395996094, 6.736352920532227, 0.4502412974834442, 0.9169650077819824, 2.9969191551208496, -0.04854787141084671, -1.270539402961731, 5.948923110961914, 0.5787533521652222, -4.5793585777282715, 2.1977946758270264, -8.525602340698242, 8.277711868286133, -0.8556555509567261, 0.2294192612171173, 11.28885269165039, 10.758705139160156, 0.579033374786377, 3.1243317127227783, 5.450899124145508, 8.410104751586914, -0.6554644703865051, 0.6763010621070862, 6.123171329498291, 1.347751498222351, 3.7724993228912354, 3.049297571182251, -5.674559593200684, -4.294704914093018, -13.206694602966309, -1.3006035089492798, 6.694187164306641, -5.54603385925293, 6.169017791748047, 4.997730731964111, -0.27421092987060547, -0.038058217614889145, 0.4141657054424286, -4.305434703826904, -1.4116519689559937, -0.2328215390443802, 3.05968976020813, 3.959064245223999, -3.7136902809143066, 4.128565788269043, -3.889352798461914, -0.8445826768875122, 1.8475267887115479, -0.11326698213815689, -0.6348041296005249, 1.3472062349319458, -2.7639951705932617, -3.2633306980133057, -1.0205705165863037, 4.806355953216553, -0.5083404183387756, 9.52260684967041, 1.0712664127349854, -14.871993064880371, -2.197768449783325, 8.325170516967773, 0.03172917664051056, -2.1891684532165527, -4.368298530578613, 6.618988990783691, 5.014122486114502, 3.3601741790771484, 2.140509605407715, 7.204531192779541, 4.106008529663086, -6.1490020751953125, 3.8422176837921143, 0.8481316566467285, -0.5515989661216736, 3.8169405460357666, 0.14193253219127655, -14.667774200439453, 7.786940574645996, 5.188600063323975, -0.612946093082428, -1.8619831800460815, 1.133242130279541, -6.421968936920166, 1.532028317451477, -1.1509853601455688, -1.7647541761398315, -4.615592956542969, -3.393181085586548, 5.531482219696045, 6.675105094909668, 2.514190435409546, -0.22768309712409973, -4.389516830444336, 3.588768482208252, 8.054680824279785, -8.072519302368164, 5.871296405792236, -10.828348159790039, -1.7000453472137451, 1.9231230020523071, -6.824935436248779, -7.584953784942627, 0.928368866443634, 4.383574962615967, -2.377265214920044, 6.7194600105285645, -3.8996102809906006, 8.771757125854492, 0.602025032043457, 1.1765165328979492, 3.074678421020508, 1.6051342487335205, 9.228901863098145, 11.032415390014648, 2.730588912963867, -0.8285238742828369, -5.8805036544799805, -3.771873712539673, 0.04624094441533089, -0.5066942572593689, -0.21836799383163452, 0.025242256000638008, 2.477698564529419, -6.753436088562012, -5.127377986907959, 5.609604358673096, 4.191462516784668, -2.678907632827759, 6.476669788360596, -3.3819563388824463, -1.2589603662490845, -9.127355575561523, 4.235611438751221, -0.04375545680522919, 5.3489155769348145, 3.546323299407959, 0.4973553419113159, 0.8300347924232483, 6.010348320007324, -0.7740601897239685, -0.7113939523696899, 0.40093761682510376, 4.093681335449219, 4.69092321395874, 9.672486305236816, -0.294709712266922, 6.44232702255249, -3.129162073135376, 3.1750874519348145, 0.7983361482620239, 8.859594345092773, 3.241692543029785, -2.0372238159179688, 3.1024467945098877, 1.949234127998352, -1.3612526655197144, -1.2452809810638428, -4.488451957702637, 14.021659851074219, 0.914506733417511, 5.1132659912109375, -5.5849528312683105, -5.184325695037842, -0.5193710327148438, -0.49635735154151917, 4.261014938354492, -4.768632411956787, 2.950432062149048, 7.827767372131348, -4.3182291984558105, 4.802310466766357, -4.5495100021362305, 1.9848123788833618, 2.0709292888641357, 8.941439628601074, -3.199138879776001, 1.0961328744888306, 5.472579479217529, -5.606134414672852, 0.2084006816148758, -4.097049713134766, -3.2570860385894775, 3.662196636199951, 3.754857063293457, 13.140530586242676, 7.137277126312256, 6.6969780921936035, -0.5838190317153931, -1.2545894384384155, 0.9301461577415466, -11.133782386779785, 8.523838996887207], "dtype": "float32"}], "e": {"__nd__": [0.03700640797615051, 0.29450857639312744, 0.9997560977935791, 0.9642823338508606, 0.0002726670354604721, 0.493496298789978, 0.04239850491285324, 0.6776859760284424, 0.0003219026839360595, 0.7700970768928528, 0.764390766620636, 0.9996212124824524, 0.8296607732772827, 0.9277569055557251, 0.0005721235647797585, 0.9999871850013733, 6.921470776433125e-05, 0.9959661960601807, 0.1611737757921219, 0.012089765630662441, 0.0362139455974102, 0.03356684371829033, 0.8699985146522522, 0.3657963275909424, 0.9993969202041626, 0.00010641902190400288, 0.49614888429641724, 0.01142705138772726, 0.9797542691230774, 2.5158224161714315e-05, 0.1520681083202362, 0.7409408688545227, 0.0005979873239994049, 0.9927973747253418, 0.709625780582428, 0.9935312271118164, 0.9918043613433838, 0.007396877743303776, 0.01818191073834896, 0.2656780183315277, 0.9767925143241882, 0.9860032796859741, 0.027965376153588295, 0.045826371759176254, 0.01817948929965496, 0.051115892827510834, 0.2333274632692337, 0.34094735980033875, 0.999966025352478, 0.07364384829998016, 0.19098511338233948, 0.9082863926887512, 0.008447759784758091, 0.13668085634708405, 0.5166419744491577, 0.9795824289321899, 0.0013858230086043477, 0.998949408531189, 0.14650416374206543, 0.6999546885490417, 0.9857330918312073, 0.9645645618438721, 0.9950141906738281, 0.7039104104042053, 0.47550293803215027, 0.8295431137084961, 0.006836786866188049, 0.9976962804794312, 0.02029392123222351, 0.0018307091668248177, 0.9999657869338989, 0.33576494455337524, 0.0027439086697995663, 0.999904990196228, 0.16130323708057404, 0.9950842261314392, 0.015558861196041107, 0.22375784814357758, 0.00021543260663747787, 7.157473737606779e-05, 0.00752349104732275, 0.4036703407764435, 0.9818350076675415, 0.9999946355819702, 0.005712809972465038, 0.9936010241508484, 0.04645490273833275, 0.3180719316005707, 0.0006072526448406279, 0.0006139834295026958, 0.01999054104089737, 0.002559059066697955, 0.49555712938308716, 0.028956620022654533, 0.9980799555778503, 0.6824596524238586, 0.695548951625824, 0.9966223239898682, 0.9302516579627991, 0.45346879959106445, 0.38850119709968567, 0.35325926542282104, 0.9954297542572021, 0.9804574847221375, 0.8617128729820251, 0.9819101095199585, 0.061649031937122345, 0.9279163479804993, 0.0008566121105104685, 0.9988144636154175, 0.6106966137886047, 0.7144232988357544, 0.952434778213501, 0.4878654181957245, 0.2191649228334427, 0.9973981380462646, 0.640780508518219, 0.010157248005270958, 0.9000512957572937, 0.00019828589574899524, 0.9997459650039673, 0.2982478141784668, 0.5571045875549316, 0.9999874830245972, 0.9999787211418152, 0.6408449411392212, 0.9578853249549866, 0.9957259297370911, 0.9997774362564087, 0.34175920486450195, 0.6629126071929932, 0.9978132843971252, 0.7937617897987366, 0.9775223135948181, 0.9547522068023682, 0.003420440712943673, 0.013457034714519978, 1.838250113905815e-06, 0.21406346559524536, 0.9987634420394897, 0.0038877325132489204, 0.9979110956192017, 0.9932920336723328, 0.431873619556427, 0.49048659205436707, 0.6020863056182861, 0.01331532746553421, 0.19597363471984863, 0.4420561194419861, 0.9551990032196045, 0.9812763333320618, 0.023806776851415634, 0.9841493368148804, 0.020048420876264572, 0.3005704879760742, 0.8638364672660828, 0.47171348333358765, 0.346422016620636, 0.793672502040863, 0.05930110439658165, 0.036850810050964355, 0.2649162709712982, 0.991888701915741, 0.37558266520500183, 0.9999268054962158, 0.7448377013206482, 3.476764902643481e-07, 0.09995105862617493, 0.999757707118988, 0.507931649684906, 0.1007273942232132, 0.01251419447362423, 0.9986670017242432, 0.9934003949165344, 0.9664364457130432, 0.8947786092758179, 0.9992573261260986, 0.9837935566902161, 0.0021310606971383095, 0.9790042638778687, 0.7001750469207764, 0.365493506193161, 0.9784783720970154, 0.5354236960411072, 4.2644833797567117e-07, 0.9995850324630737, 0.9944511651992798, 0.3513874411582947, 0.13447205722332, 0.7564367055892944, 0.0016228148015215993, 0.8223028779029846, 0.24030914902687073, 0.1461959183216095, 0.009799336083233356, 0.032509252429008484, 0.9960554838180542, 0.9987396597862244, 0.9251306653022766, 0.44332385063171387, 0.012254681438207626, 0.9731106758117676, 0.9996824860572815, 0.0003118989698123187, 0.9971886873245239, 1.982894173124805e-05, 0.15445934236049652, 0.8724862933158875, 0.0010851671686396003, 0.0005077802925370634, 0.7167442440986633, 0.9876731634140015, 0.08492285013198853, 0.9987942576408386, 0.019847886636853218, 0.9998449683189392, 0.6461194753646851, 0.7643209099769592, 0.9558360576629639, 0.8327347636222839, 0.9999018311500549, 0.9999838471412659, 0.9388076663017273, 0.3039572834968567, 0.002785596763715148, 0.02249140851199627, 0.5115581750869751, 0.3759687840938568, 0.44562390446662903, 0.506310224533081, 0.9225635528564453, 0.001165503286756575, 0.005897111725062132, 0.9963508248329163, 0.9851011633872986, 0.06422950327396393, 0.9984634518623352, 0.03286415711045265, 0.2211529165506363, 0.00010864072828553617, 0.9857354760169983, 0.4890628755092621, 0.9952691793441772, 0.9719774723052979, 0.6218376159667969, 0.6963622570037842, 0.997552752494812, 0.3156014680862427, 0.32929089665412903, 0.5989128947257996, 0.9835958480834961, 0.9909052848815918, 0.9999369978904724, 0.4268512427806854, 0.9984098076820374, 0.0419202484190464, 0.9598859548568726, 0.6896184682846069, 0.999858021736145, 0.9623734354972839, 0.11534972488880157, 0.956993579864502, 0.8753631114959717, 0.20403678715229034, 0.22351808845996857, 0.011113137938082218, 0.9999991655349731, 0.7139214873313904, 0.9940195679664612, 0.0037398876156657934, 0.005572487134486437, 0.37299931049346924, 0.3783970773220062, 0.9860882759094238, 0.008420479483902454, 0.9502838850021362, 0.9996016621589661, 0.013148275204002857, 0.9918560981750488, 0.010461777448654175, 0.8791932463645935, 0.8880453705787659, 0.999869167804718, 0.0391981415450573, 0.7495348453521729, 0.9958171844482422, 0.0036617908626794815, 0.5519124269485474, 0.016349880024790764, 0.03707309439778328, 0.9749667048454285, 0.9771314263343811, 0.9999980330467224, 0.9992057085037231, 0.9987668991088867, 0.3580543100833893, 0.22190669178962708, 0.7171049118041992, 1.4610107427870389e-05, 0.9998013377189636], "dtype": "float32"}}, {"a": [{"__nd__": [-187.38621520996094, -304.5465393066406, -124.41649627685547, 275.0650329589844, 385.0330505371094, -101.4994125366211, 141.9090576171875, -6.486037731170654, -288.64691162109375, 60.80178451538086, 17.74029541015625, -452.1461486816406, 338.9375915527344, -462.9474792480469, 78.38191223144531, -307.0543212890625, -236.51895141601562, -203.2245330810547, 218.77001953125, -369.2007141113281, -50.309898376464844, 472.26348876953125, -202.37184143066406, -3.356938600540161, 205.25433349609375, -169.1098175048828, 148.1151580810547, -192.90798950195312, 215.17318725585938, 23.82703399658203, -275.86474609375, 427.69122314453125, -23.573816299438477, -79.1166000366211, -272.6549987792969, -136.26210021972656, -24.232044219970703, 56.190521240234375, 334.0153503417969, -390.9708251953125, -100.46815490722656, 363.13946533203125, 58.74062728881836, 126.59970092773438, 481.99755859375, 105.29161834716797, 143.8884735107422, -8.315678596496582, 534.197265625, 318.8343200683594, -212.66693115234375, 168.13052368164062, -340.98675537109375, 329.4960632324219, 204.62030029296875, -45.05056381225586, -117.8733139038086, -536.046875, 231.29598999023438, -33.14076232910156, -91.70245361328125, 218.54566955566406, -198.07412719726562, -174.92579650878906, -207.44418334960938, -405.3577575683594, 723.2198486328125, -58.2293586730957, -105.15478515625, 269.8944396972656, -945.288818359375, -9.981905937194824, 2.763979434967041, -292.9046325683594, 135.94766235351562, -610.5242919921875, -2.7176780700683594, 258.2906494140625, -676.4765625, -447.4649963378906, -63.765708923339844, -25.948135375976562, -218.70388793945312, 80.20185852050781, 41.376136779785156, -439.0500183105469, -67.29792785644531, -422.651611328125, -665.3043823242188, -768.1546020507812, 261.9649353027344, -86.49188232421875, 537.9610595703125, -489.27740478515625, -157.59884643554688, 50.01116180419922, -94.78376007080078, 98.6927719116211, -7.439348220825195, -235.1265106201172, -749.2885131835938, -334.4354553222656, 467.84832763671875, 182.4895782470703, 367.0164794921875, -220.92724609375, -101.43794250488281, -305.92755126953125, 75.71819305419922, -22.053632736206055, 96.12189483642578, 489.7339172363281, -289.29278564453125, 183.45985412597656, -458.4169921875, 293.99786376953125, -169.50218200683594, -58.420143127441406, 4.336200714111328, 457.50250244140625, -45.93681716918945, -185.8343963623047, -391.8425598144531, 46.31843566894531, -293.0131530761719, -187.53878784179688, -44.16755676269531, 165.04873657226562, -251.0998077392578], "dtype": "float32"}], "e": {"__nd__": [0.0, 0.0, 0.0, 1.0, 1.0, 8.407790785948902e-45, 1.0, 0.0015222569927573204, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.4147794981338752e-22, 1.0, 0.0, 0.03366868570446968, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 5.781248743619294e-11, 4.3661352808253017e-35, 0.0, 0.0, 2.993345848967266e-11, 1.0, 1.0, 0.0, 2.382207389352189e-44, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0002445909776724875, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 2.7213770401551514e-20, 0.0, 0.0, 1.0, 4.0471547622000954e-15, 1.4932376565691683e-40, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 5.144116973222717e-26, 0.0, 1.0, 0.0, 4.622673804988153e-05, 0.9406980276107788, 0.0, 1.0, 0.0, 0.06193823739886284, 1.0, 0.0, 0.0, 2.027236748356921e-28, 5.381061915499252e-12, 0.0, 1.0, 1.0, 0.0, 5.9276328252079185e-30, 0.0, 0.0, 0.0, 1.0, 2.7356011293299575e-38, 1.0, 0.0, 0.0, 1.0, 6.85375078901268e-42, 1.0, 0.0005873229238204658, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 8.407790785948902e-45, 0.0, 1.0, 2.643802399227013e-10, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 4.2506389371250084e-26, 0.9870828986167908, 1.0, 1.121744071547779e-20, 0.0, 0.0, 1.0, 0.0, 0.0, 6.58072583604511e-20, 1.0, 0.0], "dtype": "float32"}}, {"a": [{"__nd__": [-10.0, -9.682539939880371, -9.365078926086426, -9.047618865966797, -8.730158805847168, -8.412698745727539, -8.095237731933594, -7.777777671813965, -7.460317611694336, -7.142857074737549, -6.82539701461792, -6.507936477661133, -6.190476417541504, -5.873015880584717, -5.55555534362793, -5.238095283508301, -4.920634746551514, -4.603174686431885, -4.285714149475098, -3.9682538509368896, -3.6507935523986816, -3.3333332538604736, -3.0158729553222656, -2.6984126567840576, -2.3809523582458496, -2.0634920597076416, -1.7460317611694336, -1.4285714626312256, -1.1111111640930176, -0.7936508059501648, -0.4761904776096344, -0.1587301641702652, 0.1587301641702652, 0.4761904776096344, 0.7936508059501648, 1.1111111640930176, 1.4285714626312256, 1.7460317611694336, 2.0634920597076416, 2.3809523582458496, 2.6984126567840576, 3.0158729553222656, 3.3333332538604736, 3.6507935523986816, 3.9682538509368896, 4.285714149475098, 4.603174686431885, 4.920634746551514, 5.238095283508301, 5.55555534362793, 5.873015880584717, 6.190476417541504, 6.507936477661133, 6.82539701461792, 7.142857074737549, 7.460317611694336, 7.777777671813965, 8.095237731933594, 8.412698745727539, 8.730158805847168, 9.047618865966797, 9.365078926086426, 9.682539939880371, 10.0], "dtype": "float32"}], "e": {"__nd__": [4.539786823443137e-05, 6.235901673790067e-05, 8.565657481085509e-05, 0.00011765705130528659, 0.00016161066014319658, 0.00022198056103661656, 0.0003048951330129057, 0.0004187667218502611, 0.0005751423886977136, 0.0007898659678176045, 0.00108466693200171, 0.0014893329935148358, 0.0020446612033993006, 0.002806474221870303, 0.00385103328153491, 0.005282310768961906, 0.0072416746988892555, 0.009920570999383926, 0.013576919212937355, 0.018555598333477974, 0.0253131166100502, 0.034445200115442276, 0.04671391472220421, 0.06306708604097366, 0.08463675528764725, 0.11269616335630417, 0.1485484093427658, 0.1933213621377945, 0.24766379594802856, 0.31138530373573303, 0.3831520974636078, 0.4604005813598633, 0.5395994186401367, 0.6168479323387146, 0.6886146664619446, 0.7523362040519714, 0.8066786527633667, 0.851451575756073, 0.8873038291931152, 0.9153632521629333, 0.9369329214096069, 0.9532861113548279, 0.9655547738075256, 0.974686861038208, 0.9814444184303284, 0.9864230751991272, 0.990079402923584, 0.9927583336830139, 0.9947177171707153, 0.996148943901062, 0.9971935153007507, 0.997955322265625, 0.9985106587409973, 0.9989153146743774, 0.9992101192474365, 0.9994248747825623, 0.9995812177658081, 0.9996951222419739, 0.9997780323028564, 0.9998384118080139, 0.9998823404312134, 0.9999143481254578, 0.9999376535415649, 0.9999545812606812], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(sigmoid, json.loads(_TESTS), 'cuda', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")